# Day 28 — **Profiling: cProfile & Line Profiler**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~30 min

Optimising by guesswork wastes days speeding up code that was never slow. **Profilers** tell you the truth. `cProfile` (standard library) gives per-function timings; a line-level profiler narrows it to the exact statement. We'll use `cProfile` + `pstats`, then build a tiny line-timer (since `line_profiler` isn't installed) to get the same insight keyless.

Today:
1. A deliberately slow banking aggregation.
2. `cProfile` + `pstats` — rank functions by cumulative time.
3. A line-level timing decorator (the `line_profiler` idea, from stdlib).
4. Fix the hotspot and re-measure the win.

## 1. A slow function to profile

`summarise` builds a per-customer total. It hides a classic performance bug — a linear membership test inside a loop — plus a genuinely cheap helper, so the profiler has to tell them apart.

In [1]:
def clean(name):
    return name.strip().lower()                     # cheap

def total_for(customer, txns, known):
    if customer not in known:                        # known is a LIST -> O(n) each call
        return 0.0
    return sum(t["amt"] for t in txns if clean(t["cust"]) == customer)

def summarise(txns, customers):
    known = list(customers)                          # BUG: list, not set
    return {c: total_for(c, txns, known) for c in customers}

TXNS = [{"cust": f"cust{i%50}", "amt": i * 1.0} for i in range(20_000)]
CUSTOMERS = [f"cust{i}" for i in range(50)]
res = summarise(TXNS, CUSTOMERS)
print("sample total cust0:", res["cust0"])

sample total cust0: 3990000.0


☝ It works — but is it fast? You *cannot* tell by reading; the `in known` check on a list and the repeated `clean()` calls are both suspects. Guessing which to fix is how engineers waste afternoons. Measure instead.

## 2. `cProfile` + `pstats` — rank by time

`cProfile.run` records every call; `pstats.Stats` sorts and prints the worst offenders. Sort by **cumulative time** (`cumtime`) to see where wall-time actually goes.

In [2]:
import cProfile, pstats, io

pr = cProfile.Profile()
pr.enable()
summarise(TXNS, CUSTOMERS)
pr.disable()

s = io.StringIO()
pstats.Stats(pr, stream=s).sort_stats("tottime").print_stats(5)
print("\n".join(s.getvalue().splitlines()[:12]))

         3020621 function calls (3020616 primitive calls) in 0.369 seconds

   Ordered by: internal time
   List reduced from 97 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  1000000    0.118    0.000    0.205    0.000 /var/folders/jd/b91jfh0n5r90fqg_0_2t9xg00000gn/T/ipykernel_14291/2333143035.py:1(clean)
        1    0.093    0.093    0.276    0.276 /var/folders/jd/b91jfh0n5r90fqg_0_2t9xg00000gn/T/ipykernel_14291/2333143035.py:9(summarise)
    20050    0.069    0.000    0.274    0.000 /var/folders/jd/b91jfh0n5r90fqg_0_2t9xg00000gn/T/ipykernel_14291/2333143035.py:7(<genexpr>)
  1000000    0.046    0.000    0.046    0.000 {method 'lower' of 'str' objects}
  1000000    0.042    0.000    0.042    0.000 {method 'strip' of 'str' objects}



☝ The report ranks functions by `tottime` (time *in* the function, excluding subcalls). `clean` shows a huge **ncalls** count — it runs once per transaction per customer (20,000 × 50 = 1M times). That call count, not any single slow line, is the cost. `cProfile` turns "it feels slow" into "this function is called a million times" — an actionable fact.

## 3. Line-level timing (the `line_profiler` idea)

`cProfile` is per-function. When one function has several suspect lines, you want per-line numbers. `line_profiler`'s `@profile` does this; it's not installed, so here's the same idea in ~10 stdlib lines — time named blocks with a context manager.

In [3]:
from contextlib import contextmanager
import time

BLOCKS = {}
@contextmanager
def timed(label):
    t = time.perf_counter()
    yield
    BLOCKS[label] = BLOCKS.get(label, 0) + (time.perf_counter() - t)

# instrument the two candidate lines
def summarise_instrumented(txns, customers):
    with timed("build_known"):
        known = list(customers)
    out = {}
    with timed("aggregate"):
        for c in customers:
            out[c] = total_for(c, txns, known)
    return out

BLOCKS.clear()
summarise_instrumented(TXNS, CUSTOMERS)
for label, secs in sorted(BLOCKS.items(), key=lambda x: -x[1]):
    print(f"{label:12}: {secs*1000:7.1f} ms")

aggregate   :    37.0 ms
build_known :     0.0 ms


☝ The `aggregate` block dwarfs `build_known` — confirming the loop, not the setup, is the cost. A context-manager timer is a poor man's `line_profiler` but gives the same *decision-grade* signal: which block to attack. For serious work install the real thing (`uv add --dev line_profiler`, decorate with `@profile`, run `kernprof -l`); the mental model is identical.

## 4. Fix the hotspot, re-measure

The profile pointed at two things: the `in list` check (O(n)) and the per-transaction `clean()`. Fix both — use a `set` for membership and pre-clean once — then re-time to *prove* the win.

In [4]:
def summarise_fast(txns, customers):
    known = set(customers)                           # O(1) membership
    totals = dict.fromkeys(customers, 0.0)
    for t in txns:
        c = clean(t["cust"])                         # clean each txn ONCE
        if c in known:
            totals[c] += t["amt"]
    return totals

t = time.perf_counter(); summarise(TXNS, CUSTOMERS);      slow = time.perf_counter() - t
t = time.perf_counter(); summarise_fast(TXNS, CUSTOMERS); fast = time.perf_counter() - t
assert summarise(TXNS, CUSTOMERS) == summarise_fast(TXNS, CUSTOMERS)   # same result
print(f"slow: {slow*1000:6.1f} ms")
print(f"fast: {fast*1000:6.1f} ms   ({slow/fast:.1f}x faster, identical output)")

slow:   37.2 ms
fast:    1.6 ms   (23.7x faster, identical output)


☝ The rewrite cleans each transaction once and uses a `set` for O(1) lookups — the same result, several times faster, and the `assert` proves correctness didn't change. That last part is the discipline: an optimisation that changes the answer is a bug, not a speed-up. **Profile → fix hotspot → assert same output → measure.**

## 5. Recap — profile before you optimise

| Tool | Granularity | Use when |
|---|---|---|
| `cProfile` + `pstats` | per-function (calls, tottime, cumtime) | "which function is slow?" |
| `line_profiler` / `timed` CM | per-line / per-block | "which *line* in the hot function?" |
| `time.perf_counter` | ad-hoc A/B timing | proving a fix actually helped |
| `assert old == new` | correctness | every optimisation, always |

The profiling loop is the same at every scale — from this function to the LangGraph agent in today's other notebook: **measure, find the hotspot, fix only that, prove the output is unchanged.** Intuition about performance is usually wrong; the profiler is always right.